In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/featured_data.csv",
                 parse_dates=["StartDatetime"])
df = df.sort_values("StartDatetime").reset_index(drop=True)

cat_cols = ["ChargerCompany", "ChargerType", "Location", "TimeOfDay"]
num_cols = ["Hour", "Day", "Month", "DayOfWeek", "IsWeekend",
            "User_PrevSessions", "User_AvgPastDemand",
            "Charger_PrevSessions", "Charger_AvgPastDemand", "IsNewUser"]

X = df[cat_cols + num_cols]
y = df["HighDemand"]

split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

HighDemand
1    0.503055
0    0.496945
Name: proportion, dtype: float64
HighDemand
0    0.510056
1    0.489944
Name: proportion, dtype: float64


In [2]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
    "SVM": SVC(probability=True, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, learning_rate=0.1, random_state=42),
    "LightGBM": LGBMClassifier(n_estimators=200, learning_rate=0.1, random_state=42, verbose=-1),
}
slow = {"KNN", "SVM"}

results, fitted = [], {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    if name in slow:
        idx = X_train.sample(15000, random_state=42).index
        pipe.fit(X_train.loc[idx], y_train.loc[idx])
    else:
        pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    results.append({"Model": name,
                    "Accuracy": accuracy_score(y_test, pred),
                    "Precision": precision_score(y_test, pred),
                    "Recall": recall_score(y_test, pred),
                    "F1": f1_score(y_test, pred),
                    "ROC_AUC": roc_auc_score(y_test, proba)})
    fitted[name] = pipe
    print("done:", name)

results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False).round(4)
results_df

done: Logistic Regression
done: Decision Tree
done: Random Forest
done: KNN
done: SVM
done: XGBoost
done: LightGBM


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
2,Random Forest,0.7285,0.6973,0.7876,0.7397,0.8091
6,LightGBM,0.7302,0.6998,0.7868,0.7408,0.8070
5,XGBoost,0.7240,0.7045,0.7522,0.7276,0.8057
0,Logistic Regression,0.7114,0.7018,0.7145,0.7081,0.7888
4,SVM,0.7127,0.6780,0.7878,0.7288,0.7821
1,Decision Tree,0.7001,0.6899,0.7047,0.6972,0.7801
3,KNN,0.6941,0.6733,0.7298,0.7004,0.7623


In [4]:
import os, joblib
os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)

results_df.to_csv("../results/classification_results.csv", index=False)
best = results_df.iloc[0]["Model"]
joblib.dump(fitted[best], "../models/best_classification_model.pkl")
print("Best:", best)

Best: Random Forest
